# Model Training in Vertex

In [1]:
from google.cloud import aiplatform

In [2]:
PROJECT_ID = "dw-analytics-d01"
REGION = "us"
ZONE = "central1"
LOCATION = f"{REGION}-{ZONE}"

In [4]:
aiplatform.init(project=PROJECT_ID, location=LOCATION)

In [5]:
segment = 106
bucket = f"dev_dw_npii_adhoc/ml/dm-propensity/models/{segment}"

DISPLAY_NAME = (f"prop-model-{segment}-training")
SCRIPT_PATH="./vertex_task.py"
STAGING_BUCKET = (bucket)
MODEL_TRAINING_IMAGE = f"{REGION}-docker.pkg.dev/vertex-ai/training/scikit-learn-cpu.0-23:latest"
MODEL_SERVING_IMAGE = f"{REGION}-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.0-23:latest"
BASE_OUTPUT_DIR = f"gs://{bucket}/vertex_ai"

In [6]:
custom_training_job = aiplatform.CustomTrainingJob(
    project=PROJECT_ID,
    location=LOCATION,
    display_name=DISPLAY_NAME,
    script_path=SCRIPT_PATH,
    staging_bucket=STAGING_BUCKET,
    container_uri=MODEL_TRAINING_IMAGE,
    # requirements=REQUIREMENTS,
    model_serving_container_image_uri=MODEL_SERVING_IMAGE,
)

In [7]:
MACHINE_TYPE = "n1-standard-4"
MODEL_DISPLAY_NAME = (
    f"prop-{segment}-model"
)

In [8]:
model = custom_training_job.run(
    machine_type=MACHINE_TYPE,
    model_display_name=MODEL_DISPLAY_NAME,
    args=[f"--segment={segment}",],
    base_output_dir=BASE_OUTPUT_DIR,
    service_account="dev-ana-ainb-admin@dw-analytics-d01.iam.gserviceaccount.com",
)

Training script copied to:
gs://dev_dw_npii_adhoc/ml/dm-propensity/models/106/aiplatform-2022-06-29-17:54:19.231-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://dev_dw_npii_adhoc/ml/dm-propensity/models/106/vertex_ai 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/7885253466589757440?project=134453458552
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/2031391987659177984?project=134453458552
CustomTrainingJob projects/134453458552/locations/us-central1/trainingPipelines/7885253466589757440 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/134453458552/locations/us-central1/trainingPipelines/7885253466589757440 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/134453458552/locations/us-central1/trainingPipelines/7885253466589757440 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob proje

In [10]:
MODEL_RESOURCE_NAME = model.resource_name

# Deployment to Endpoint

In [11]:
ENDPOINT_DISPLAY_NAME = f"pc-{segment}-endpoint"
MACHINE_TYPE_SERVING = "n1-standard-2"

In [12]:
endpoint = aiplatform.Endpoint.create(
    display_name=ENDPOINT_DISPLAY_NAME,
    location=LOCATION,
    project=PROJECT_ID,
)

Creating Endpoint
Create Endpoint backing LRO: projects/134453458552/locations/us-central1/endpoints/9108908478356783104/operations/6525895698930466816
Endpoint created. Resource name: projects/134453458552/locations/us-central1/endpoints/9108908478356783104
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/134453458552/locations/us-central1/endpoints/9108908478356783104')


In [13]:
model = aiplatform.Model(model_name=MODEL_RESOURCE_NAME)

In [14]:
model.deploy(
    endpoint=endpoint,
    deployed_model_display_name=MODEL_DISPLAY_NAME,
    machine_type=MACHINE_TYPE_SERVING,
    traffic_percentage=100,
    min_replica_count=1,
    max_replica_count=2,
    accelerator_type=None,
    accelerator_count=None,
)

Deploying model to Endpoint : projects/134453458552/locations/us-central1/endpoints/9108908478356783104
Deploy Endpoint model backing LRO: projects/134453458552/locations/us-central1/endpoints/9108908478356783104/operations/1182374741055373312
Endpoint model deployed. Resource name: projects/134453458552/locations/us-central1/endpoints/9108908478356783104


resource name: projects/134453458552/locations/us-central1/endpoints/9108908478356783104

In [5]:
import pandas as pd
from google.cloud import bigquery
from google.cloud import bigquery_storage
from google.cloud import storage

import pickle

bq_client = bigquery.Client(project='dw-bq-data-d00')
bqstorageclient = bigquery_storage.BigQueryReadClient()

In [6]:
query = f"""SELECT A_A8642_HM_MKT_VAL, BUYS_Q_01, BUYS_Q_04, BUYS_Q_08, NUM_PERIODS,
PH_MREDEEM730D_PERC, PH_PFREQ365D, TOTAL_TXNS_L12M,
is_in_trade_area, A_A9350N_ECONOMIC_STB_01_10
FROM `dw-bq-data-d00.SANDBOX_ANALYTICS.dm_pc_refresh_eval_data_w_margin`
WHERE em_segment = {segment}
ORDER BY COUPON_BARCODE DESC
LIMIT 3"""

In [7]:
to_pred = bq_client.query(query).result().to_dataframe(
    bqstorage_client=bqstorageclient)

to_pred

,A_A8642_HM_MKT_VAL,BUYS_Q_01,BUYS_Q_04,BUYS_Q_08,NUM_PERIODS,PH_MREDEEM730D_PERC,PH_PFREQ365D,TOTAL_TXNS_L12M,is_in_trade_area,A_A9350N_ECONOMIC_STB_01_10
0,225000,0,0,0,6,0.333333,12,0E-9,1,1.000000000
1,225000,0,0,0,10,0.000000,12,0E-9,1,1.000000000
2,1250000,7,0,0,7,0.186047,11,11.000000000,1,1.000000000


In [8]:
storage_client = storage.Client(project='dw-bq-data-d00')
bucket_name = 'dev_dw_npii_adhoc'

source_file = f'ml/dm-propensity/scalers/{segment}/vertex_ai/scaler_{segment}.pkl'
filename = f'scaler_{segment}.pkl'

bcket = storage_client.bucket(bucket_name)
blob = bcket.blob(source_file)
blob.download_to_filename(filename)

with open(filename, 'rb') as scaler:
    scaler = pickle.load(scaler)

/opt/conda/envs/pipe_env/lib/python3.6/site-packages/sklearn/base.py:315: UserWarning: Trying to unpickle estimator StandardScaler from version 0.23.1 when using version 0.24.2. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)


In [9]:
to_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   A_A8642_HM_MKT_VAL           3 non-null      int64  
 1   BUYS_Q_01                    3 non-null      int64  
 2   BUYS_Q_04                    3 non-null      int64  
 3   BUYS_Q_08                    3 non-null      int64  
 4   NUM_PERIODS                  3 non-null      int64  
 5   PH_MREDEEM730D_PERC          3 non-null      float64
 6   PH_PFREQ365D                 3 non-null      int64  
 7   TOTAL_TXNS_L12M              3 non-null      object 
 8   is_in_trade_area             3 non-null      int64  
 9   A_A9350N_ECONOMIC_STB_01_10  3 non-null      object 
dtypes: float64(1), int64(7), object(2)
memory usage: 368.0+ bytes


In [10]:
to_pred['A_A9350N_ECONOMIC_STB_01_10'] = to_pred['A_A9350N_ECONOMIC_STB_01_10'].astype('int')
cat_features = ['is_in_trade_area', 'A_A9350N_ECONOMIC_STB_01_10']
cat_feat = to_pred[cat_features].copy()

# to_pred.pop('TARGET_14')
to_pred.drop(columns=cat_features, inplace=True)
    
to_pred = pd.DataFrame(scaler.transform(to_pred),
                       columns=to_pred.columns,
                       index=to_pred.index,                       
                      )

to_pred[cat_features] = cat_feat

In [11]:
to_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   A_A8642_HM_MKT_VAL           3 non-null      float64
 1   BUYS_Q_01                    3 non-null      float64
 2   BUYS_Q_04                    3 non-null      float64
 3   BUYS_Q_08                    3 non-null      float64
 4   NUM_PERIODS                  3 non-null      float64
 5   PH_MREDEEM730D_PERC          3 non-null      float64
 6   PH_PFREQ365D                 3 non-null      float64
 7   TOTAL_TXNS_L12M              3 non-null      float64
 8   is_in_trade_area             3 non-null      int64  
 9   A_A9350N_ECONOMIC_STB_01_10  3 non-null      int64  
dtypes: float64(8), int64(2)
memory usage: 368.0 bytes


In [12]:
instances = to_pred.to_numpy()
instances

array([[-0.85438931, -0.8974801 , -1.00772989, -0.85612015, -0.20257811,
        -0.16009265,  0.05680523, -0.52900093,  1.        ,  1.        ],
       [-0.85438931, -0.8974801 , -1.00772989, -0.85612015,  1.50182359,
        -1.51963307,  0.05680523, -0.52900093,  1.        ,  1.        ],
       [ 2.35540225,  2.18932236, -1.00772989, -0.85612015,  0.22352232,
        -0.76081706, -0.18368682,  0.22907254,  1.        ,  1.        ]])

In [13]:
array = [list(x) for x in instances]
array

[[-0.8543893094241147,
  -0.8974800964734979,
  -1.0077298894293323,
  -0.8561201463110412,
  -0.2025781100371106,
  -0.16009264785113758,
  0.05680523215027506,
  -0.5290009257343274,
  1.0,
  1.0],
 [-0.8543893094241147,
  -0.8974800964734979,
  -1.0077298894293323,
  -0.8561201463110412,
  1.501823591848596,
  -1.5196330701426128,
  0.05680523215027506,
  -0.5290009257343274,
  1.0,
  1.0],
 [2.355402246763874,
  2.189322357389406,
  -1.0077298894293323,
  -0.8561201463110412,
  0.22352231543431608,
  -0.7608170604884169,
  -0.1836868239911111,
  0.22907253564492444,
  1.0,
  1.0]]

In [24]:
# endpoint = aiplatform.Endpoint('9108908478356783104')

endpoint.predict(
    instances=array
)

Prediction(predictions=[0.0, 0.0, 1.0], deployed_model_id='6494265429458944000', explanations=None)

In [23]:
# array_2 = [[
#     0.7896502681355868, -0.015536538226954015, 0.5620315525992154,
#     0.37937914887373675, 1.0757231663771696, 1.5393359389822157,
#     -0.4241788801324973, 1.7452194584034282, 1.0, 1.0
# ],
#     [
#     -0.00637611690421424, -0.015536538226954015, 0.0393526341206262,
#     0.37937914887373675, -0.20257811003711063, -0.5443113912732525,
#     0.29729728829166124, 1.262809073889359, 1.0, 1.0
# ],
#     [
#     0.7896502681355868, 1.7483505782661337, 0.5620315525992154,
#     0.7912122472686628, 1.5018235918485963, 0.768377861674467,
#     0.29729728829166124, 0.9871459970241764, 0.0, 1.0
# ]]